# Drawing Recognition Model

Train digit + letter + shape recognition. Export to TensorFlow.js for Next.js deployment.

In [ ]:
# Cell 1 — Imports and GPU/MPS Setup
# Configure device BEFORE importing TensorFlow (for Apple Silicon Metal)
import os
import random

# Apple Silicon: enable Metal plugin for GPU acceleration
if os.environ.get("TF_METAL") is None:
    os.environ["TF_METAL"] = "1"

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import sklearn
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import albumentations as A
from tqdm import tqdm
import json
import time
import tensorflow_datasets as tfds

# --- Device detection and report ---
print("=" * 60)
print("ENVIRONMENT REPORT")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Keras version:     {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")
# Check for Metal (Apple) — may show as GPU on Mac
all_devices = tf.config.list_physical_devices()
print(f"\nDevices: {[d.device_type for d in all_devices]}")
if gpus:
    for gpu in gpus:
        print(f"  GPU: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
else:
    print("  No GPU found — using CPU (or Metal on Apple Silicon)")
print("=" * 60)

# --- Reproducibility: set all seeds to 42 ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# TF deterministic behavior (may slow training slightly)
tf.config.experimental.enable_op_determinism()

print("Random seeds set to 42. Ready for data loading.")

In [ ]:
# Cell 2 — Data Collection and Loading
# Load MNIST, EMNIST (digits, byclass), and Quick Draw; merge into 72-class dataset

# --- 1. MNIST (70k digits, 28x28) ---
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = keras.datasets.mnist.load_data()
mnist_x = np.concatenate([mnist_x_train, mnist_x_test], axis=0)
mnist_y = np.concatenate([mnist_y_train, mnist_y_test], axis=0)
print("1. MNIST:")
print(f"   Shape: {mnist_x.shape}, labels 0-9, total {len(mnist_y)} samples")
print(f"   Class distribution: {np.bincount(mnist_y)}")

# --- 2. EMNIST Digits (280k digits, 28x28) ---
ds_digits = tfds.load("emnist", name="digits", split="train+test", as_supervised=True)
emnist_digits_x, emnist_digits_y = [], []
for img, label in tqdm(ds_digits, desc="EMNIST Digits"):
    # EMNIST image: (28,28,1), may need transpose for display; we'll preprocess later
    emnist_digits_x.append(np.squeeze(img.numpy()))
    emnist_digits_y.append(label.numpy())
emnist_digits_x = np.stack(emnist_digits_x, axis=0)
emnist_digits_y = np.array(emnist_digits_y, dtype=np.int32)
print("\n2. EMNIST Digits:")
print(f"   Shape: {emnist_digits_x.shape}, labels 0-9, total {len(emnist_digits_y)} samples")
print(f"   Class distribution: {np.bincount(emnist_digits_y)}")

# --- 3. EMNIST ByClass (62 classes: 10 digits + 26 upper + 26 lower) ---
ds_byclass = tfds.load("emnist", name="byclass", split="train+test", as_supervised=True)
emnist_byclass_x, emnist_byclass_y = [], []
for img, label in tqdm(ds_byclass, desc="EMNIST ByClass"):
    emnist_byclass_x.append(np.squeeze(img.numpy()))
    emnist_byclass_y.append(label.numpy())
emnist_byclass_x = np.stack(emnist_byclass_x, axis=0)
emnist_byclass_y = np.array(emnist_byclass_y, dtype=np.int32)
print("\n3. EMNIST ByClass:")
print(f"   Shape: {emnist_byclass_x.shape}, labels 0-61, total {len(emnist_byclass_y)} samples")
print(f"   Classes 0-9 (digits): {np.sum((emnist_byclass_y >= 0) & (emnist_byclass_y < 10))}")
print(f"   Classes 10-35 (upper): {np.sum((emnist_byclass_y >= 10) & (emnist_byclass_y < 36))}")
print(f"   Classes 36-61 (lower): {np.sum((emnist_byclass_y >= 36) & (emnist_byclass_y < 62))}")

# --- 4. Quick Draw: 10 shape classes, 10k samples each (28x28) ---
# Use classes that exist in Quick Draw: circle, square, triangle, star, line, zigzag, hexagon, diamond, sun, moon
QUICKDRAW_SHAPES = ["circle", "square", "triangle", "star", "line", "zigzag", "hexagon", "diamond", "sun", "moon"]
SAMPLES_PER_SHAPE = 10_000

qd_info = tfds.builder("quickdraw_bitmap").info
label_names = qd_info.features["label"].names
# Pick 10 class indices: prefer our shape names, then fill from label_names
chosen_indices = []
for s in QUICKDRAW_SHAPES:
    if s in label_names and len(chosen_indices) < 10:
        chosen_indices.append(label_names.index(s))
while len(chosen_indices) < 10:
    for name in label_names:
        idx = label_names.index(name)
        if idx not in chosen_indices:
            chosen_indices.append(idx)
            break
    if len(chosen_indices) >= 10:
        break
chosen_indices = chosen_indices[:10]
qd_label_to_our = {idx: (62 + i) for i, idx in enumerate(chosen_indices)}  # our classes 62-71

ds_qd = tfds.load("quickdraw_bitmap", split="train", as_supervised=True)
count_per_class = {our: 0 for our in range(62, 72)}
qd_x_list, qd_y_list = [], []
for img, label in tqdm(ds_qd, desc="Quick Draw"):
    lab = label.numpy()
    if lab in qd_label_to_our:
        our_label = qd_label_to_our[lab]
        if count_per_class[our_label] < SAMPLES_PER_SHAPE:
            qd_x_list.append(np.squeeze(img.numpy()))
            qd_y_list.append(our_label)
            count_per_class[our_label] += 1
    if all(c >= SAMPLES_PER_SHAPE for c in count_per_class.values()):
        break
quickdraw_x = np.stack(qd_x_list, axis=0)
quickdraw_y = np.array(qd_y_list, dtype=np.int32)
chosen_shape_names = [label_names[i] for i in chosen_indices]
print("\n4. Quick Draw (10 shapes):")
print(f"   Shape: {quickdraw_x.shape}, labels 62-71, total {len(quickdraw_y)} samples")
print(f"   Classes: {chosen_shape_names}")
print(f"   Per-class counts: {count_per_class}")

# --- Merge into unified label space: 0-9 digits, 10-35 upper, 36-61 lower, 62-71 shapes ---
# Digits: merge MNIST + EMNIST Digits + EMNIST ByClass digits (0-9)
# Letters: EMNIST ByClass 10-61 (already 10-35 upper, 36-61 lower)
# Shapes: Quick Draw 62-71

def merge_digits_and_letters():
    x_mnist = mnist_x[..., np.newaxis]  # (N,28,28,1)
    y_mnist = mnist_y
    x_ed = emnist_digits_x[..., np.newaxis]
    y_ed = emnist_digits_y
    # ByClass: digits 0-9 and letters 10-61
    mask_digit = emnist_byclass_y < 10
    mask_letter = emnist_byclass_y >= 10
    x_bc_d = emnist_byclass_x[mask_digit][..., np.newaxis]
    y_bc_d = emnist_byclass_y[mask_digit]
    x_bc_l = emnist_byclass_x[mask_letter][..., np.newaxis]
    y_bc_l = emnist_byclass_y[mask_letter]
    x_digits = np.concatenate([x_mnist, x_ed, x_bc_d], axis=0)
    y_digits = np.concatenate([y_mnist, y_ed, y_bc_d], axis=0)
    x_letters = x_bc_l
    y_letters = y_bc_l
    return x_digits, y_digits, x_letters, y_letters

x_digits, y_digits, x_letters, y_letters = merge_digits_and_letters()
x_shapes = quickdraw_x[..., np.newaxis]
y_shapes = quickdraw_y

X_all = np.concatenate([x_digits, x_letters, x_shapes], axis=0)
y_all = np.concatenate([y_digits, y_letters, y_shapes], axis=0)

# Build label list for later (index -> name)
DIGIT_NAMES = [str(i) for i in range(10)]
UPPER_NAMES = [chr(ord("A") + i) for i in range(26)]
LOWER_NAMES = [chr(ord("a") + i) for i in range(26)]
ALL_LABELS = DIGIT_NAMES + UPPER_NAMES + LOWER_NAMES + chosen_shape_names
assert len(ALL_LABELS) == 72

print("\n" + "=" * 60)
print("FINAL COMBINED DATASET")
print("=" * 60)
print(f"Total samples: {len(X_all)}")
print(f"Total classes: 72 (digits 0-9, uppercase 10-35, lowercase 36-61, shapes 62-71)")
print(f"Image shape: {X_all.shape[1:]}")
print(f"Class distribution (first 15): {np.bincount(y_all, minlength=72)[:15]}")
print("=" * 60)

In [ ]:
# Cell 3 — Exploratory Data Analysis (EDA)

# Helper: get 2D image for display (support (28,28) or (28,28,1))
def to_display(img):
    return np.squeeze(img)

# 1. 10×10 grid of random samples, labeled with class
fig, axes = plt.subplots(10, 10, figsize=(12, 12))
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_all), size=100, replace=False)
for i, ax in enumerate(axes.flat):
    ax.imshow(to_display(X_all[idx[i]]), cmap="gray")
    ax.set_title(ALL_LABELS[y_all[idx[i]]], fontsize=8)
    ax.axis("off")
plt.suptitle("Random samples from combined dataset (100)", fontsize=14)
plt.tight_layout()
plt.show()

# 2. Class distribution bar chart (72 classes) — check for imbalance
counts = np.bincount(y_all, minlength=72)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(72), counts, color="steelblue", edgecolor="navy", alpha=0.8)
ax.set_xlabel("Class index")
ax.set_ylabel("Sample count")
ax.set_title("Class distribution (72 classes)")
ax.set_xticks(range(0, 72, 4))
ax.set_xticklabels([f"{i}\n{ALL_LABELS[i]}" for i in range(0, 72, 4)], fontsize=7)
plt.tight_layout()
plt.show()

# 3. Average pixel intensity image per class ("average 3", "average A", etc.)
# Show one row per group: digits (0-9), uppercase (10-17), lowercase (36-43), shapes (62-71)
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
for row, (start, title) in enumerate([
    (0, "Digits 0-9"),
    (10, "Uppercase A-H"),
    (36, "Lowercase a-h"),
    (62, "Shapes"),
]):
    for col in range(10):
        c = start + col
        if c < 72:
            mask = y_all == c
            if np.any(mask):
                mean_img = np.mean(X_all[mask], axis=0)
                axes[row, col].imshow(to_display(mean_img), cmap="gray")
            axes[row, col].set_title(ALL_LABELS[c], fontsize=9)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(title, fontsize=10)
plt.suptitle("Average pixel intensity per class", fontsize=14)
plt.tight_layout()
plt.show()

# 4. Pixel intensity histogram across full dataset
plt.figure(figsize=(10, 4))
plt.hist(X_all.flatten(), bins=50, color="steelblue", edgecolor="black", alpha=0.7)
plt.xlabel("Pixel intensity")
plt.ylabel("Count")
plt.title("Pixel intensity histogram (full dataset)")
plt.tight_layout()
plt.show()

# 5. Sample count per dataset source (Digits / Letters / Shapes)
n_digits = np.sum(y_all < 10)
n_letters = np.sum((y_all >= 10) & (y_all < 62))
n_shapes = np.sum(y_all >= 62)
source_counts = [n_digits, n_letters, n_shapes]
source_names = ["Digits\n(MNIST+EMNIST+ByClass)", "Letters\n(EMNIST ByClass)", "Shapes\n(Quick Draw)"]
plt.figure(figsize=(6, 4))
plt.bar(source_names, source_counts, color=["#2ecc71", "#3498db", "#e74c3c"], edgecolor="black")
plt.ylabel("Sample count")
plt.title("Samples per dataset source")
for i, v in enumerate(source_counts):
    plt.text(i, v + 0.01 * max(source_counts), str(v), ha="center", fontsize=11)
plt.tight_layout()
plt.show()

# 6. Five most likely confused class pairs (visually similar)
confused_pairs = [
    ("0", "O", "digit zero vs letter O"),
    ("1", "l", "digit one vs lowercase L"),
    ("1", "I", "digit one vs letter I"),
    ("5", "S", "digit five vs letter S"),
    ("q", "g", "lowercase q vs g"),
]
print("Five most likely confused class pairs (visually similar):")
print("-" * 60)
for a, b, desc in confused_pairs:
    idx_a = ALL_LABELS.index(a) if a in ALL_LABELS else -1
    idx_b = ALL_LABELS.index(b) if b in ALL_LABELS else -1
    if idx_a >= 0 and idx_b >= 0:
        print(f"  {a!r} (class {idx_a}) <-> {b!r} (class {idx_b}): {desc}")
print("-" * 60)

In [ ]:
# Cell 4 — Preprocessing Pipeline
# Single-image preprocessing (numpy) then apply via tf.data.map to full dataset
from PIL import Image

INPUT_SIZE = 32
PAD = 2

def preprocess_image(img):
    """Apply full preprocessing: resize 32x32, normalize [0,1], center, white-on-black, contrast, channel."""
    img = np.squeeze(img)
    if img.dtype != np.float32:
        img = img.astype(np.float32) / 255.0
    # Invert if needed: we want drawn=1, background=0 (white-on-black)
    if np.mean(img) > 0.5:
        img = 1.0 - img
    # Bounding box of non-zero pixels; use threshold to ignore noise
    thresh = 0.1
    rows = np.any(img > thresh, axis=1)
    cols = np.any(img > thresh, axis=0)
    if not (np.any(rows) and np.any(cols)):
        # Empty or no stroke: use full image
        rmin, rmax, cmin, cmax = 0, img.shape[0] - 1, 0, img.shape[1] - 1
    else:
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
    crop = img[rmin : rmax + 1, cmin : cmax + 1]
    h, w = crop.shape
    # Center crop in 28x28 with 2px padding (effective inner 24x24)
    canvas = np.zeros((28, 28), dtype=np.float32)
    top = max(0, (28 - h) // 2)
    left = max(0, (28 - w) // 2)
    bottom = min(28, top + h)
    right = min(28, left + w)
    canvas[top:bottom, left:right] = crop[: bottom - top, : right - left]
    # Resize to 32x32
    pil_img = Image.fromarray((canvas * 255).astype(np.uint8))
    pil_img = pil_img.resize((INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)
    img_out = np.array(pil_img, dtype=np.float32) / 255.0
    # Contrast normalization: stretch to [0, 1]
    mn, mx = img_out.min(), img_out.max()
    if mx - mn > 1e-8:
        img_out = (img_out - mn) / (mx - mn)
    # Add channel: (32, 32, 1)
    img_out = np.expand_dims(img_out, axis=-1)
    return img_out.astype(np.float32)

# Apply to full dataset via tf.data pipeline (.map)
def preprocess_tf(x, y):
    x = tf.numpy_function(preprocess_image, [x], tf.float32)
    x.set_shape((INPUT_SIZE, INPUT_SIZE, 1))
    return x, y

ds_full = tf.data.Dataset.from_tensor_slices((X_all, y_all))
ds_full = ds_full.map(preprocess_tf, num_parallel_calls=tf.data.AUTOTUNE)
# Materialize into numpy for stratified split (tf.data doesn't do stratified)
print("Preprocessing full dataset (tf.data.map)...")
X_processed = np.stack([x.numpy() for x, _ in tqdm(ds_full, total=len(X_all))], axis=0)
y_processed = y_all

# One-hot for training
y_onehot = keras.utils.to_categorical(y_processed, num_classes=72)

# Stratified split: 80% train, 10% val, 10% test
X_train, X_rest, y_train, y_rest = train_test_split(
    X_processed, y_onehot, test_size=0.2, stratify=y_processed, random_state=SEED
)
y_rest_labels = np.argmax(y_rest, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest_labels, random_state=SEED
)

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Train samples: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
train_labels = np.argmax(y_train, axis=1)
val_labels = np.argmax(y_val, axis=1)
test_labels = np.argmax(y_test, axis=1)
print(f"Train class distribution (first 12): {np.bincount(train_labels, minlength=72)[:12]}")
print("=" * 60)

In [ ]:
# Cell 5 — Aggressive Data Augmentation
# Apply ONLY to training data. No horizontal flip (preserve d/b, p/q, 6/9).

def build_augment():
    return A.Compose([
        A.Rotate(limit=15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5),
        A.Affine(shear=5, p=0.4),
        A.ElasticTransform(alpha=1, sigma=3, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.GaussNoise(var_limit=(5.0, 25.0), p=0.3),
        A.GridDistortion(num_steps=3, distort_limit=0.2, p=0.3),
    ])

def maybe_erosion_dilation(img):
    """Random erosion or dilation (thin/thick strokes). img: (H,W) uint8."""
    if np.random.random() > 0.5:
        return img
    k = 2
    kernel = np.ones((k, k), np.uint8)
    if np.random.random() > 0.5:
        return cv2.erode(img, kernel)
    return cv2.dilate(img, kernel)

def augment_image(img_np):
    """img_np: (32, 32, 1) float32 [0,1]. Returns (32, 32, 1) float32."""
    img = np.squeeze(img_np)
    img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    t = build_augment()
    out = t(image=img_uint8)["image"]
    if np.random.random() < 0.4:
        out = maybe_erosion_dilation(out)
    out = out.astype(np.float32) / 255.0
    return np.expand_dims(out, axis=-1)

# Visualize: 1 original + 8 augmented (generic sample)
fig, axes = plt.subplots(1, 9, figsize=(14, 2))
sample_idx = np.random.default_rng(SEED).integers(0, len(X_train))
orig = X_train[sample_idx]
axes[0].imshow(np.squeeze(orig), cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")
for i in range(8):
    aug = augment_image(orig)
    axes[i + 1].imshow(np.squeeze(aug), cmap="gray")
    axes[i + 1].set_title(f"Aug {i+1}")
    axes[i + 1].axis("off")
plt.suptitle("Augmentation samples (1 original + 8 augmented)")
plt.tight_layout()
plt.show()

# Visualize for tricky class 'q'
q_class_idx = ALL_LABELS.index("q")
q_train_idx = np.where(np.argmax(y_train, axis=1) == q_class_idx)[0]
if len(q_train_idx) > 0:
    q_idx = q_train_idx[0]
    fig, axes = plt.subplots(1, 9, figsize=(14, 2))
    orig_q = X_train[q_idx]
    axes[0].imshow(np.squeeze(orig_q), cmap="gray")
    axes[0].set_title("Original 'q'")
    axes[0].axis("off")
    for i in range(8):
        aug = augment_image(orig_q)
        axes[i + 1].imshow(np.squeeze(aug), cmap="gray")
        axes[i + 1].set_title(f"Aug {i+1}")
        axes[i + 1].axis("off")
    plt.suptitle("Augmentation for class 'q'")
    plt.tight_layout()
    plt.show()
else:
    print("No 'q' sample in train set for visualization.")

# Training dataset: shuffle → augment → batch(128) → prefetch
def augment_map(x, y):
    x = tf.numpy_function(augment_image, [x], tf.float32)
    x.set_shape((INPUT_SIZE, INPUT_SIZE, 1))
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(10000, seed=SEED).map(augment_map, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(128).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(256).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(256).prefetch(tf.data.AUTOTUNE)

print("Pipeline ready:")
print(f"  train_ds: shuffle(10000) → augment → batch(128) → prefetch")
print(f"  val_ds:   batch(256) → prefetch")
print(f"  test_ds:  batch(256) → prefetch")

In [ ]:
# Cell 6 — Model Architecture
# ResNet-style CNN with Squeeze-and-Excitation blocks (Keras Functional API)

def se_block(x, channels, ratio=8, name="se"):
    """Squeeze-and-Excitation: GlobalAvgPool → Dense(reduced) → ReLU → Dense(channels) → Sigmoid → scale."""
    reduced = max(channels // ratio, 8)
    se = keras.layers.GlobalAveragePooling2D(name=f"{name}_gap")(x)
    se = keras.layers.Dense(reduced, activation="relu", name=f"{name}_fc1")(se)
    se = keras.layers.Dense(channels, activation="sigmoid", name=f"{name}_fc2")(se)
    se = keras.layers.Reshape((1, 1, channels))(se)
    return keras.layers.Multiply(name=f"{name}_scale")([x, se])

def residual_block(x, filters, se_ratio, pool=True, name="res"):
    """Conv→BN→ReLU→Conv→BN → SE → (+shortcut) → ReLU → optional MaxPool."""
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = keras.layers.Conv2D(filters, 1, strides=1, name=f"{name}_shortcut")(x)
    y = keras.layers.Conv2D(filters, 3, padding="same", name=f"{name}_conv1")(x)
    y = keras.layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = keras.layers.ReLU(name=f"{name}_relu1")(y)
    y = keras.layers.Conv2D(filters, 3, padding="same", name=f"{name}_conv2")(y)
    y = keras.layers.BatchNormalization(name=f"{name}_bn2")(y)
    y = se_block(y, filters, ratio=se_ratio, name=f"{name}_se")
    y = keras.layers.Add(name=f"{name}_add")([shortcut, y])
    y = keras.layers.ReLU(name=f"{name}_relu2")(y)
    if pool:
        y = keras.layers.MaxPooling2D(2, name=f"{name}_pool")(y)
    return y

# Input
inputs = keras.layers.Input(shape=(INPUT_SIZE, INPUT_SIZE, 1), name="input")

# Stem
x = keras.layers.Conv2D(64, 3, padding="same", name="stem_conv1")(inputs)
x = keras.layers.BatchNormalization(name="stem_bn1")(x)
x = keras.layers.ReLU(name="stem_relu1")(x)
x = keras.layers.Conv2D(64, 3, padding="same", name="stem_conv2")(x)
x = keras.layers.BatchNormalization(name="stem_bn2")(x)
x = keras.layers.ReLU(name="stem_relu2")(x)
x = keras.layers.MaxPooling2D(2, name="stem_pool")(x)  # (16, 16, 64)

# Residual Block 1 (×2) → output (8, 8, 128)
x = residual_block(x, 128, se_ratio=16, pool=True, name="res1a")   # (8, 8, 128)
x = residual_block(x, 128, se_ratio=16, pool=False, name="res1b")  # (8, 8, 128)

# Residual Block 2 (×3) → output (4, 4, 256)
x = residual_block(x, 256, se_ratio=16, pool=True, name="res2a")  # (4, 4, 256)
x = residual_block(x, 256, se_ratio=16, pool=False, name="res2b")  # (4, 4, 256)
x = residual_block(x, 256, se_ratio=16, pool=False, name="res2c") # (4, 4, 256)

# Residual Block 3 (×2)
x = residual_block(x, 512, se_ratio=16, pool=False, name="res3a")
x = residual_block(x, 512, se_ratio=16, pool=False, name="res3b")

# Head
x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
x = keras.layers.Dense(1024, name="fc1")(x)
x = keras.layers.BatchNormalization(name="head_bn1")(x)
x = keras.layers.ReLU(name="head_relu1")(x)
x = keras.layers.Dropout(0.4, name="drop1")(x)
x = keras.layers.Dense(512, name="fc2")(x)
x = keras.layers.BatchNormalization(name="head_bn2")(x)
x = keras.layers.ReLU(name="head_relu2")(x)
x = keras.layers.Dropout(0.3, name="drop2")(x)
outputs = keras.layers.Dense(72, activation="softmax", name="output")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="drawing_recognition_cnn")
model.summary()

# Plot architecture (saved to file)
try:
    keras.utils.plot_model(model, "model_architecture.png", show_shapes=True, show_layer_names=True, dpi=96)
    plt.figure(figsize=(14, 10))
    plt.imshow(plt.imread("model_architecture.png"))
    plt.axis("off")
    plt.title("Model architecture")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Plot skipped:", e)

In [ ]:
# Cell 7 — Training Configuration
# AdamW, label smoothing, cosine LR with warmup, callbacks

# Optimizer: AdamW
optimizer = keras.optimizers.AdamW(
    learning_rate=0.001,
    weight_decay=1e-4,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
)

# Loss: Categorical Crossentropy with label smoothing
loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# Metrics: accuracy, top-3 accuracy
metrics = [
    "accuracy",
    keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy"),
]

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

# Learning rate schedule: warmup 5 epochs (1e-6 → 0.001), then cosine decay to 1e-6 over 50 epochs, restart every 50 with half LR
def cosine_with_warmup(epoch, warmup=5, peak_lr=0.001, decay_epochs=50, min_lr=1e-6):
    if epoch < warmup:
        return float(1e-6 + (peak_lr - 1e-6) * (epoch / warmup))
    cycle = (epoch - warmup) % decay_epochs
    restart_factor = 0.5 ** ((epoch - warmup) // decay_epochs)
    lr_max = peak_lr * restart_factor
    return float(min_lr + 0.5 * (lr_max - min_lr) * (1 + np.cos(np.pi * cycle / decay_epochs)))

lr_schedule = keras.callbacks.LearningRateScheduler(cosine_with_warmup, verbose=0)

# Custom callback: progress table every epoch
class ProgressTableCallback(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.start_time = None
        self.best_val_acc = 0.0

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        print(f"{'Epoch':<6} {'loss':<10} {'acc':<8} {'top3':<8} {'val_loss':<10} {'val_acc':<8} {'val_top3':<8} {'lr':<12} {'time':<8}")
        print("-" * 90)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        elapsed = time.time() - self.start_time
        lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
        val_acc = logs.get("val_accuracy", 0)
        self.best_val_acc = max(self.best_val_acc, val_acc)
        print(f"{epoch+1:<6} {logs.get('loss', 0):<10.4f} {logs.get('accuracy', 0):<8.4f} {logs.get('top_3_accuracy', 0):<8.4f} "
              f"{logs.get('val_loss', 0):<10.4f} {logs.get('val_accuracy', 0):<8.4f} {logs.get('val_top_3_accuracy', 0):<8.4f} "
              f"{lr:<12.2e} {elapsed:<8.1f}s")

callbacks = [
    keras.callbacks.ModelCheckpoint("best_model.h5", monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    keras.callbacks.TensorBoard("./logs"),
    keras.callbacks.CSVLogger("training_log.csv"),
    lr_schedule,
    ProgressTableCallback(),
]

EPOCHS = 100
BATCH_SIZE = 128  # already set in train_ds
print("Training config: AdamW, label_smoothing=0.1, cosine warmup+decay, 100 epochs max, batch 128")
print("Callbacks: ModelCheckpoint(best_model.h5), EarlyStopping(15), ReduceLROnPlateau, TensorBoard, CSVLogger, LR schedule, Progress table")

In [ ]:
# Cell 8 — Train the Model

class TenEpochSummary(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best_val = 0.0

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            logs = logs or {}
            lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            best = max(getattr(self, "best_val", 0), logs.get("val_accuracy", 0))
            self.best_val = best
            print(f"  [Epoch {epoch+1}] train_acc={logs.get('accuracy', 0):.4f} val_acc={logs.get('val_accuracy', 0):.4f} best_val={best:.4f} lr={lr:.2e}")

train_start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks + [TenEpochSummary()],
    verbose=1,
)
train_elapsed = time.time() - train_start

# Best epoch and val accuracy from history
val_accs = history.history["val_accuracy"]
best_val_acc = max(val_accs)
best_epoch_1based = val_accs.index(best_val_acc) + 1
epochs_run = len(val_accs)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Total training time:     {train_elapsed / 60:.1f} min")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Epoch of best model:     {best_epoch_1based}")
print(f"Epochs run (before early stop): {epochs_run}")
print("=" * 60)

In [ ]:
# Cell 9 — Evaluation and Error Analysis

model = keras.models.load_model("best_model.h5", compile=True)
test_results = model.evaluate(test_ds)
test_accuracy = test_results[1]
test_top3_accuracy = test_results[2]
print(f"1. Test accuracy: {test_accuracy:.4f}")
print(f"2. Test top-3 accuracy: {test_top3_accuracy:.4f}")

# Predictions for full test set
y_pred_proba = model.predict(test_ds)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

# 3. Per-class accuracy (sorted ascending — worst first)
per_class_correct = np.zeros(72)
per_class_total = np.zeros(72)
for c in range(72):
    mask = y_true == c
    per_class_total[c] = np.sum(mask)
    per_class_correct[c] = np.sum((y_true == c) & (y_pred == c))
per_class_acc = np.where(per_class_total > 0, per_class_correct / per_class_total, 0)
worst_first = np.argsort(per_class_acc)
print("\n3. Per-class accuracy (worst 15 first):")
print(f"   {'Class':<8} {'Name':<6} {'Acc':<8} {'Count'}")
for i in worst_first[:15]:
    print(f"   {i:<8} {ALL_LABELS[i]:<6} {per_class_acc[i]:<8.4f} {int(per_class_total[i])}")

# 4. Classification report
print("\n4. Classification report:")
print(classification_report(y_true, y_pred, target_names=ALL_LABELS, zero_division=0))

# 5. Confusion matrix heatmap (72×72), highlight top 20 misclassifications
cm = confusion_matrix(y_true, y_pred)
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)
flat = cm_off_diag.flatten()
top20_flat_idx = np.argsort(-flat)[:20]
top20_ij = [np.unravel_index(k, cm.shape) for k in top20_flat_idx]

fig, ax = plt.subplots(figsize=(14, 12))
ax.imshow(cm, cmap="Blues")
for (i, j) in top20_ij:
    if i != j and cm[i, j] > 0:
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="red", linewidth=2))
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(72))
ax.set_yticks(range(72))
ax.set_xticklabels(ALL_LABELS, fontsize=5)
ax.set_yticklabels(ALL_LABELS, fontsize=5)
plt.title("Confusion matrix (red = top 20 off-diagonal errors)")
plt.tight_layout()
plt.show()

# 6. Top 10 confusion pairs
off_diag_pairs = []
for i in range(72):
    for j in range(72):
        if i != j and cm[i, j] > 0:
            pct = 100 * cm[i, j] / per_class_total[i] if per_class_total[i] > 0 else 0
            off_diag_pairs.append((i, j, cm[i, j], pct))
off_diag_pairs.sort(key=lambda x: -x[2])
print("\n6. Top 10 confusion pairs (true → predicted):")
for i, j, count, pct in off_diag_pairs[:10]:
    print(f"   {ALL_LABELS[i]!r} → {ALL_LABELS[j]!r}: {count} times ({pct:.2f}%)")

# 7. Five examples of mistakes
mistake_idx = np.where(y_pred != y_true)[0]
np.random.default_rng(SEED).shuffle(mistake_idx)
show_mistakes = mistake_idx[:5]
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for ax, idx in zip(axes, show_mistakes):
    ax.imshow(np.squeeze(X_test[idx]), cmap="gray")
    ax.set_title(f"True: {ALL_LABELS[y_true[idx]]}\nPred: {ALL_LABELS[y_pred[idx]]}")
    ax.axis("off")
plt.suptitle("5 example misclassifications")
plt.tight_layout()
plt.show()

# 8. Twenty examples correct with >99.9% confidence
max_proba = np.max(y_pred_proba, axis=1)
correct = y_pred == y_true
high_conf = correct & (max_proba > 0.999)
high_conf_idx = np.where(high_conf)[0]
np.random.default_rng(SEED).shuffle(high_conf_idx)
show_high = high_conf_idx[:20]
fig, axes = plt.subplots(4, 5, figsize=(12, 8))
for ax, idx in zip(axes.flat, show_high):
    ax.imshow(np.squeeze(X_test[idx]), cmap="gray")
    ax.set_title(f"{ALL_LABELS[y_true[idx]]} ({max_proba[idx]*100:.1f}%)")
    ax.axis("off")
plt.suptitle("20 correct predictions with >99.9% confidence")
plt.tight_layout()
plt.show()

# 9. Accuracy and loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"], label="train acc")
ax1.plot(history.history["val_accuracy"], label="val acc")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.set_title("Accuracy vs epoch")
ax2.plot(history.history["loss"], label="train loss")
ax2.plot(history.history["val_loss"], label="val loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.set_title("Loss vs epoch")
plt.tight_layout()
plt.show()

# Target benchmark
print("\n" + "=" * 60)
min_class_acc = np.min(per_class_acc[per_class_total > 0]) if np.any(per_class_total > 0) else 0
if test_accuracy >= 0.995 and min_class_acc >= 0.98:
    print("TARGET MET: Overall >99.5%, all classes ≥98%")
else:
    print("TARGET NOT MET. Diagnosis:")
    if test_accuracy < 0.995:
        print(f"  - Overall test accuracy {test_accuracy:.2%} < 99.5%")
    if min_class_acc < 0.98:
        worst_class = worst_first[0]
        print(f"  - Class {ALL_LABELS[worst_class]!r} accuracy {per_class_acc[worst_class]:.2%} < 98%")
    print("  Consider: more data for weak classes, fine-tuning (Cell 10), or stronger augmentation.")
print("=" * 60)

In [ ]:
# Cell 10 — Fine-Tuning Pass (if accuracy < 99.5%)
# Oversample worst 10 classes 3×, train 30 epochs at lr=1e-4, batch 64, save as best_model_finetuned.h5

if test_accuracy >= 0.995:
    print("Test accuracy already ≥99.5%. Skipping fine-tuning.")
else:
    # 1. Bottom 10 worst-performing classes (from per_class_acc, worst_first)
    worst_10_classes = worst_first[:10]
    print("1. Worst 10 classes (by accuracy):", [ALL_LABELS[c] for c in worst_10_classes])

    # 2. Oversample those classes by 3× in training data
    train_labels_flat = np.argmax(y_train, axis=1)
    X_extra, y_extra = [], []
    for c in worst_10_classes:
        mask = train_labels_flat == c
        X_c = X_train[mask]
        y_c = y_train[mask]
        # Add 2× more copies (3× total)
        for _ in range(2):
            X_extra.append(X_c)
            y_extra.append(y_c)
    if X_extra:
        X_extra = np.concatenate(X_extra, axis=0)
        y_extra = np.concatenate(y_extra, axis=0)
        X_train_ft = np.concatenate([X_train, X_extra], axis=0)
        y_train_ft = np.concatenate([y_train, y_extra], axis=0)
    else:
        X_train_ft, y_train_ft = X_train, y_train

    # Shuffle
    shuffle_idx = np.random.default_rng(SEED).permutation(len(X_train_ft))
    X_train_ft = X_train_ft[shuffle_idx]
    y_train_ft = y_train_ft[shuffle_idx]
    print(f"2. Oversampled train size: {len(X_train_ft)} (was {len(X_train)})")

    # 3. New datasets: batch 64, same augmentation
    train_ds_ft = tf.data.Dataset.from_tensor_slices((X_train_ft, y_train_ft))
    train_ds_ft = train_ds_ft.shuffle(10000, seed=SEED).map(augment_map, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds_ft = train_ds_ft.batch(64).prefetch(tf.data.AUTOTUNE)

    # Reload best model and recompile with smaller LR
    model_ft = keras.models.load_model("best_model.h5", compile=True)
    model_ft.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")],
    )

    # 4. Train 30 epochs
    history_ft = model_ft.fit(
        train_ds_ft,
        validation_data=val_ds,
        epochs=30,
        callbacks=[
            keras.callbacks.ModelCheckpoint("best_model_finetuned.h5", monitor="val_accuracy", save_best_only=True, verbose=1),
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1),
        ],
        verbose=1,
    )

    # 5. Re-evaluate on test set and on worst 10 classes
    test_results_ft = model_ft.evaluate(test_ds)
    print(f"\n5. After fine-tuning — Test accuracy: {test_results_ft[1]:.4f} (before: {test_accuracy:.4f})")

    y_pred_ft = np.argmax(model_ft.predict(test_ds), axis=1)
    print("   Per-class accuracy on worst 10 classes (before → after):")
    for c in worst_10_classes:
        mask = y_true == c
        n = np.sum(mask)
        acc_before = per_class_correct[c] / n if n > 0 else 0
        acc_after = np.sum((y_true == c) & (y_pred_ft == c)) / n if n > 0 else 0
        print(f"     {ALL_LABELS[c]!r}: {acc_before:.4f} → {acc_after:.4f}")

    model_ft.save("best_model_finetuned.h5")
    print("   Saved: best_model_finetuned.h5")